# 04 — Real MIMIC Stage 2: Offline FQI Alarm Timing

This notebook is the real-data controller stage.

It uses the **frozen Stage-1 risk score** and learns:

```text
risk p(t)
risk trend Δp(t)
time within stay
trust ρ(t)
        ↓
   FQI controller
        ↓
   WAIT / ALARM
```

The notebook intentionally requires the Stage-1 risk files and the original hourly rows.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression

ROOT = Path.cwd().resolve().parents[1]
DATA = ROOT / "sae_trust_aware_v2" / "data"

print("ROOT:", ROOT)
print("DATA:", DATA)

train = pd.read_parquet(DATA / "train_hourly.parquet")
val = pd.read_parquet(DATA / "val_hourly.parquet")
test = pd.read_parquet(DATA / "test_hourly.parquet")

BASE_FEATURES = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "gcs"
]

FEATURES = []

for c in BASE_FEATURES:
    FEATURES.extend([
        c,
        f"{c}_delta",
        f"{c}_mean3h"
    ])

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

ROOT: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
DATA: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data
Train: (793, 30)
Validation: (338, 30)
Test: (453, 30)


## Real-data transition construction

For MIMIC, build one trajectory per ICU stay.

For each hour `t`:

```text
state = [risk_t, risk_delta_t, t/T, trust]
action = WAIT or ALARM
reward = derived from:
    - SAE onset window
    - clinician-response/trust model
    - lead time
next_state = next hour
```

**Do not train an online DQN.** The controller is trained only from retrospective transitions.


In [2]:
train_m = train.dropna(subset=FEATURES + ["y"]).copy()
val_m = val.dropna(subset=FEATURES + ["y"]).copy()
test_m = test.dropna(subset=FEATURES + ["y"]).copy()

X_train = train_m[FEATURES]
y_train = train_m["y"]

X_val = val_m[FEATURES]
y_val = val_m["y"]

X_test = test_m[FEATURES]

clf = HistGradientBoostingClassifier(
    max_depth=3,
    max_iter=300,
    learning_rate=0.05,
    random_state=42
)

clf.fit(X_train, y_train)

raw_val = clf.predict_proba(X_val)[:, 1]
raw_train = clf.predict_proba(X_train)[:, 1]
raw_test = clf.predict_proba(X_test)[:, 1]

calibrator = IsotonicRegression(
    out_of_bounds="clip"
)

calibrator.fit(raw_val, y_val)

train_m["risk"] = calibrator.transform(raw_train)
val_m["risk"] = calibrator.transform(raw_val)
test_m["risk"] = calibrator.transform(raw_test)

print("Frozen Stage-1 risk generated.")
print("Train risk range:", train_m["risk"].min(), train_m["risk"].max())
print("Test risk range:", test_m["risk"].min(), test_m["risk"].max())

Frozen Stage-1 risk generated.
Train risk range: 0.0 0.05759162303664921
Test risk range: 0.0 0.05759162303664921


## Build retrospective policy transitions

MIMIC does not contain historical alarms from our proposed system. Therefore this notebook constructs an **explicit behavior-policy alarm log over real patient trajectories**. This is an offline policy-simulation baseline, not historical clinician action data. Keep this distinction in the manuscript.

The behavior policy is a risk threshold plus small exploration; the FQI learner never sees the test set.


In [3]:
def prepare_trajectory(df):
    df = df.sort_values(
        ["subject_id", "stay_id", "hour"]
    ).copy()

    df["risk_delta"] = (
        df.groupby(["subject_id", "stay_id"])["risk"]
        .diff()
        .fillna(0.0)
    )

    df["t_frac"] = (
        df.groupby(["subject_id", "stay_id"])["hour"]
        .transform(
            lambda x: (x - x.min()) / max(1, x.max() - x.min())
        )
    )

    return df


train_risk = prepare_trajectory(train_m)
val_risk = prepare_trajectory(val_m)
test_risk = prepare_trajectory(test_m)

print("Train risk:", train_risk.shape)
print("Validation risk:", val_risk.shape)
print("Test risk:", test_risk.shape)

Train risk: (372, 33)
Validation risk: (254, 33)
Test risk: (379, 33)


### Final training rule

For the publication run, create `stage1_train_risk.parquet` and then:

```text
TRAIN risk trajectories → FQI training
VALIDATION trajectories → threshold / operating-point selection
TEST trajectories → one final evaluation
```

Do not train FQI on validation or test trajectories in the final experiment.


In [4]:
WAIT = 0
ALARM = 1

HORIZON = 6
R_HIT = 3.0
R_FALSE_ALARM = 1.0
R_MISS = 2.0

TRUST_FA_DECAY = 0.28
TRUST_RECOVERY = 0.015
TRUST_TRUE_RECOVERY = 0.04

rng = np.random.default_rng(42)


def build_transitions(df, trust_aware=True):
    transitions = []

    for (subject_id, stay_id), g in df.groupby(
        ["subject_id", "stay_id"],
        sort=False
    ):
        g = g.sort_values("hour").reset_index(drop=True)

        trust = 1.0

        for i in range(len(g) - 1):

            row = g.iloc[i]
            next_row = g.iloc[i + 1]

            risk = float(row["risk"])
            risk_delta = float(row["risk_delta"])
            t_frac = float(row["t_frac"])

            # Trust is part of the state only for trust-aware FQI.
            if trust_aware:
                state = np.array(
                    [risk, risk_delta, t_frac, trust],
                    dtype=np.float32
                )
            else:
                state = np.array(
                    [risk, risk_delta, t_frac],
                    dtype=np.float32
                )

            # Historical behavior is not available for this hypothetical
            # alarm system, so construct a transparent behavior policy.
            if risk >= 0.03:
                action = ALARM if rng.random() < 0.75 else WAIT
            else:
                action = ALARM if rng.random() < 0.10 else WAIT

            future = g[
                (g["hour"] > row["hour"]) &
                (g["hour"] <= row["hour"] + HORIZON)
            ]

            true_future = bool((future["y"] == 1).any())

            if action == ALARM:

                if true_future:
                    reward = R_HIT
                    trust = min(
                        1.0,
                        trust + TRUST_TRUE_RECOVERY
                    )
                else:
                    reward = -R_FALSE_ALARM * (
                        1.0 + 0.5 * trust
                    )
                    trust = max(
                        0.0,
                        trust - TRUST_FA_DECAY
                    )

                done = True
                next_state = np.zeros_like(state)

            else:

                if true_future:
                    reward = -R_MISS / HORIZON
                else:
                    reward = 0.05

                if trust_aware:
                    next_state = np.array(
                        [
                            float(next_row["risk"]),
                            float(next_row["risk_delta"]),
                            float(next_row["t_frac"]),
                            trust
                        ],
                        dtype=np.float32
                    )
                else:
                    next_state = np.array(
                        [
                            float(next_row["risk"]),
                            float(next_row["risk_delta"]),
                            float(next_row["t_frac"])
                        ],
                        dtype=np.float32
                    )

                done = False

            transitions.append(
                (
                    state,
                    action,
                    reward,
                    next_state,
                    done
                )
            )

    return transitions


transitions_fixed = build_transitions(
    train_risk,
    trust_aware=False
)

transitions_trust = build_transitions(
    train_risk,
    trust_aware=True
)

print("Fixed transitions:", len(transitions_fixed))
print("Trust-aware transitions:", len(transitions_trust))

Fixed transitions: 359
Trust-aware transitions: 359


In [5]:
sys.path.insert(
    0,
    str(ROOT / "sae_trust_aware_v2" / "src")
)

from sae_core import FQIController, Config

cfg = Config(
    horizon=6,
    gamma=0.98,
    fqi_iters=15,
    n_trees=80,
    min_samples_leaf=5,
    seed=42
)

fixed_controller = FQIController(cfg)

fixed_controller.fit(
    transitions_fixed,
    log=True
)

trust_controller = FQIController(cfg)

trust_controller.fit(
    transitions_trust,
    log=True
)

print("FQI training completed.")

FQI training completed.


In [6]:
def evaluate_controller(df, controller, trust_aware=True):

    total_events = 0
    detected_events = 0
    false_alarms = 0
    total_hours = 0

    leads = []
    alarm_count = 0

    for (subject_id, stay_id), g in df.groupby(
        ["subject_id", "stay_id"],
        sort=False
    ):
        g = g.sort_values("hour").reset_index(drop=True)

        total_hours += len(g)

        event_hours = g.loc[
            g["y"] == 1,
            "hour"
        ].values

        # SAE event exists if any future-SAE label exists.
        event_exists = len(event_hours) > 0

        if event_exists:
            total_events += 1

        trust = 1.0
        alarmed = False

        for i in range(len(g)):

            row = g.iloc[i]

            if trust_aware:
                state = np.array(
                    [
                        float(row["risk"]),
                        float(row["risk_delta"]),
                        float(row["t_frac"]),
                        float(trust)
                    ],
                    dtype=np.float32
                )
            else:
                state = np.array(
                    [
                        float(row["risk"]),
                        float(row["risk_delta"]),
                        float(row["t_frac"])
                    ],
                    dtype=np.float32
                )

            action = controller.act(state)

            if action == ALARM:

                alarm_count += 1
                alarmed = True

                future_events = event_hours[
                    event_hours > row["hour"]
                ]

                if len(future_events) > 0:
                    detected_events += 1

                    lead = (
                        future_events[0]
                        - row["hour"]
                    )

                    leads.append(float(lead))

                    trust = min(
                        1.0,
                        trust + TRUST_TRUE_RECOVERY
                    )

                else:
                    false_alarms += 1

                    trust = max(
                        0.0,
                        trust - TRUST_FA_DECAY
                    )

                break

    sensitivity = (
        detected_events / total_events
        if total_events > 0 else 0
    )

    false_alarm_rate = (
        100 * false_alarms / total_hours
        if total_hours > 0 else 0
    )

    alarm_rate = (
        100 * alarm_count / total_hours
        if total_hours > 0 else 0
    )

    return {
        "events": total_events,
        "detected": detected_events,
        "sensitivity": sensitivity,
        "median_lead": np.median(leads) if leads else 0,
        "false_alarms_per_100h": false_alarm_rate,
        "alarms_per_100h": alarm_rate,
        "false_alarms": false_alarms,
        "alarms": alarm_count
    }

In [7]:
fixed_results = evaluate_controller(
    test_risk,
    fixed_controller,
    trust_aware=False
)

trust_results = evaluate_controller(
    test_risk,
    trust_controller,
    trust_aware=True
)

results = pd.DataFrame([
    {
        "Model": "Fixed FQI",
        **fixed_results
    },
    {
        "Model": "Trust-Aware FQI",
        **trust_results
    }
])

display(results.round(4))

,Model,events,detected,sensitivity,median_lead,false_alarms_per_100h,alarms_per_100h,false_alarms,alarms
0,Fixed FQI,3,2,0.6667,18.5,1.3193,1.847,5,7
1,Trust-Aware FQI,3,3,1.0000,14.0,1.0554,1.847,4,7


In [8]:
print("========================================")
print("FINAL REAL-DATA RL RESULTS")
print("========================================")

display(
    results[
        [
            "Model",
            "sensitivity",
            "median_lead",
            "false_alarms_per_100h",
            "alarms_per_100h"
        ]
    ].round(4)
)

print("\nIMPORTANT:")
print("These results are from the current MIMIC cohort.")
print("The cohort is small, so results should be treated as preliminary.")

FINAL REAL-DATA RL RESULTS


,Model,sensitivity,median_lead,false_alarms_per_100h,alarms_per_100h
0,Fixed FQI,0.6667,18.5,1.3193,1.847
1,Trust-Aware FQI,1.0000,14.0,1.0554,1.847



IMPORTANT:
These results are from the current MIMIC cohort.
The cohort is small, so results should be treated as preliminary.


In [9]:
OUT = DATA / "mimic_fqi_results.csv"

results.to_csv(
    OUT,
    index=False
)

print("Results saved to:")
print(OUT)

Results saved to:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_trust_aware_v2/data/mimic_fqi_results.csv


In [10]:
print("""
========================================
SAE TRUST-AWARE RL PIPELINE COMPLETE
========================================

Stage 1:
- Supervised SAE risk forecasting
- Validation calibration
- Held-out test evaluation

Stage 2:
- Offline Fitted-Q Iteration
- Fixed FQI baseline
- Trust-aware FQI
- WAIT / ALARM actions
- Event sensitivity
- Lead time
- False-alarm burden

No patient-level train/test overlap was permitted.
""")


SAE TRUST-AWARE RL PIPELINE COMPLETE

Stage 1:
- Supervised SAE risk forecasting
- Validation calibration
- Held-out test evaluation

Stage 2:
- Offline Fitted-Q Iteration
- Fixed FQI baseline
- Trust-aware FQI
- WAIT / ALARM actions
- Event sensitivity
- Lead time
- False-alarm burden

No patient-level train/test overlap was permitted.



## What this real-data run currently proves

It proves that the **pipeline can be executed on real MIMIC trajectories** using a frozen Stage-1 risk stream and an explicit retrospective behavior policy.

It does **not** by itself prove clinical superiority, because MIMIC does not provide the clinician's response to our hypothetical alarms. The trust-response model must therefore be treated as a model assumption and evaluated with sensitivity analysis until response/override logs are available.


## Evaluation protocol

1. Train controller on TRAIN transitions.
2. Use VALIDATION to select operating point / matched threshold.
3. Evaluate exactly once on TEST.
4. Report:
   - event sensitivity
   - effective/acted-upon sensitivity
   - median lead time
   - false alarms per 100 patient-hours
   - alarms per patient-day
   - confidence intervals
   - matched-burden threshold baseline
   - conventional classifier baseline
   - trust trajectory / sensitivity analysis

Do not report a synthetic result as a MIMIC result.
